# Crystal Structure Generator using Variational Autoencoder (VAE)

Pipeline:
1. Load CIF dataset
2. Extract structural features
3. Train VAE model
4. Generate new crystal parameters
5. Convert generated parameters to CIF


In [1]:
!pip install pymatgen torch pandas numpy scikit-learn

## Import Libraries

In [2]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
from pymatgen.core import Structure, Lattice

## Load Dataset

In [3]:
dataset = pd.read_csv('train.csv')
dataset.head()

,Unnamed: 0,material_id,cif,energy_per_atom
0,9251,C-130499-1826-36,# generated using pymatgen\ndata_C\n_symmetry_...,-154.311891
1,6412,C-13904-4247-31,# generated using pymatgen\ndata_C\n_symmetry_...,-154.332183
2,3301,C-92138-4782-35,# generated using pymatgen\ndata_C\n_symmetry_...,-154.178157
3,4777,C-192672-505-73,# generated using pymatgen\ndata_C\n_symmetry_...,-154.309339
4,3764,C-193956-5355-22,# generated using pymatgen\ndata_C\n_symmetry_...,-154.141835


## Extract Crystal Features

In [4]:
features = []
targets = []

cif_column = [col for col in dataset.columns if 'cif' in col.lower()][0]

for i in range(len(dataset)):
    try:
        cif_string = dataset[cif_column][i]
        structure = Structure.from_str(cif_string, fmt='cif')

        num_atoms = len(structure)
        volume = structure.volume
        a, b, c = structure.lattice.abc
        alpha, beta, gamma = structure.lattice.angles

        features.append([num_atoms, volume, a, b, c, alpha, beta, gamma])
        targets.append(dataset['energy_per_atom'][i])
    except:
        continue

X = np.array(features)
y = np.array(targets)

print('Dataset shape:', X.shape)

C:\Users\Dell\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\pymatgen\core\structure.py:3109: UserWarning: Issues encountered while parsing CIF: 1 fractional coordinates rounded to ideal values to avoid issues with finite precision.
  struct = parser.parse_structures(primitive=primitive)[0]


C:\Users\Dell\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\pymatgen\core\structure.py:3109: UserWarning: Issues encountered while parsing CIF: 2 fractional coordinates rounded to ideal values to avoid issues with finite precision.
  struct = parser.parse_structures(primitive=primitive)[0]


Dataset shape: (6091, 8)


## Normalize Data

In [5]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_tensor = torch.tensor(X_scaled, dtype=torch.float32)

## Define VAE Model

In [6]:
class CrystalVAE(nn.Module):
    def __init__(self, input_dim=8, latent_dim=3):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU()
        )

        self.mu = nn.Linear(32, latent_dim)
        self.logvar = nn.Linear(32, latent_dim)

        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 64),
            nn.ReLU(),
            nn.Linear(64, input_dim)
        )

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, x):
        h = self.encoder(x)
        mu = self.mu(h)
        logvar = self.logvar(h)
        z = self.reparameterize(mu, logvar)
        recon = self.decoder(z)
        return recon, mu, logvar

## Loss Function

In [7]:
def vae_loss(recon_x, x, mu, logvar):
    recon_loss = nn.MSELoss()(recon_x, x)
    kl_loss = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
    return recon_loss + kl_loss

## Train Model

In [8]:
model = CrystalVAE()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for epoch in range(300):
    optimizer.zero_grad()

    recon, mu, logvar = model(X_tensor)
    loss = vae_loss(recon, X_tensor, mu, logvar)

    loss.backward()
    optimizer.step()

    if epoch % 20 == 0:
        print('Epoch:', epoch, 'Loss:', loss.item())

Epoch: 0 Loss: 1.0274600982666016
Epoch: 20 Loss: 1.0025209188461304


Epoch: 40 Loss: 1.0002622604370117
Epoch: 60 Loss: 0.9996020793914795


Epoch: 80 Loss: 0.9884558916091919
Epoch: 100 Loss: 0.9491932392120361


Epoch: 120 Loss: 0.9267327189445496
Epoch: 140 Loss: 0.9251408576965332


Epoch: 160 Loss: 0.9214119911193848
Epoch: 180 Loss: 0.9165792465209961


Epoch: 200 Loss: 0.9191384315490723
Epoch: 220 Loss: 0.9188838005065918


Epoch: 240 Loss: 0.9190207719802856
Epoch: 260 Loss: 0.9150334000587463


Epoch: 280 Loss: 0.9169500470161438


## Generate New Crystals

In [9]:
with torch.no_grad():
    z = torch.randn(5, 3)
    generated = model.decoder(z).numpy()

generated = scaler.inverse_transform(generated)
print(generated)

[[ 9.631161  61.25769    2.7670584  4.380308   6.063365  90.39422
  89.97661   90.13276  ]
 [ 7.5098114 47.32168    2.7900574  3.9356024  5.2872586 97.32265
  96.42044   97.2899   ]
 [ 9.8053875 62.524117   2.7703698  4.4121685  6.1031795 91.5618
  91.12422   90.689064 ]
 [ 9.205443  58.693886   2.786654   4.3176813  5.8777895 87.971214
  88.49766   87.689644 ]
 [14.20181   89.48619    2.7429037  5.249046   7.2536526 90.98859
  91.36029   90.66047  ]]

## Convert Generated Parameters to CIF

In [10]:
num_atoms, volume, a, b, c, alpha, beta, gamma = generated[0]

lattice = Lattice.from_parameters(a, b, c, alpha, beta, gamma)

coords = np.random.rand(int(max(1, round(num_atoms))), 3)
species = ['C'] * len(coords)

structure = Structure(lattice, species, coords)
structure.to(filename='generated_crystal.cif')

print('Crystal saved as generated_crystal.cif')

Crystal saved as generated_crystal.cif


In [11]:
!python generate.py

Generated sample: tensor([[ 0.0340, -0.6086,  0.1838, -0.0213, -1.5360, -1.0196,  0.6443,  0.5539]])


In [ ]:
!python generate.py

Generated sample: tensor([[ 1.1173, -0.5636, -1.2243,  0.1625,  0.6702,  1.7162, -0.9669,  1.6453]])
